In [ ]:
# Watershed Segmentation
# 
# This notebook implements Watershed segmentation on all 80 preprocessed images.
# 
# Watershed overview:
# - Treats gradient magnitude as topographic surface
# - Floods from markers (sure foreground/background) to find boundaries
# - Requires marker generation: we use Otsu thresholding + distance transform
# 
# We test different distance transform thresholds and morphological parameters.

In [ ]:
# Part 1: Setup

import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import json
import time

# Import preprocessing module
from preprocessing import load_image_pair

# Configure matplotlib
plt.rcParams['figure.figsize'] = (15, 5)

print("environment setup complete")

# Define paths
PROJECT_DATA = Path('../project_data')
PREPROCESSED_DIR = PROJECT_DATA / 'preprocessed'
RESULTS_DIR = Path('../results/watershed')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load manifest
df_manifest = pd.read_csv(PROJECT_DATA / 'selected_images.csv')

print(f"total images: {len(df_manifest)}")
print(f"results directory: {RESULTS_DIR}")

In [ ]:
# Part 2: Watershed Implementation

def segment_watershed(image, dist_threshold=0.5, morph_kernel_size=3, morph_iterations=2):
    """
    Segment image using marker-based Watershed algorithm.
    
    Pipeline:
    1. Convert to grayscale
    2. Otsu thresholding for initial binary segmentation
    3. Morphological opening to remove noise
    4. Distance transform to find sure foreground
    5. Dilation to find sure background
    6. Create markers and apply watershed
    
    Args:
        image: input RGB image (H x W x 3), uint8
        dist_threshold: fraction of max distance for sure foreground (0.0-1.0)
        morph_kernel_size: kernel size for morphological operations
        morph_iterations: iterations for opening operation
    
    Returns:
        mask: binary segmentation mask (0 or 255)
        markers: watershed markers for debugging
    """
    # Step 1: Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    
    # Step 2: Otsu's thresholding
    # Try both BINARY and BINARY_INV, pick the one where border is mostly 0
    ret, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    ret_inv, thresh_inv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Pick version where borders are mostly 0 (background)
    border_sum = (thresh[0, :].sum() + thresh[-1, :].sum() + 
                  thresh[:, 0].sum() + thresh[:, -1].sum())
    border_sum_inv = (thresh_inv[0, :].sum() + thresh_inv[-1, :].sum() + 
                      thresh_inv[:, 0].sum() + thresh_inv[:, -1].sum())
    
    if border_sum_inv < border_sum:
        thresh = thresh_inv
    
    # Step 3: Noise removal with morphological opening
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (morph_kernel_size, morph_kernel_size)
    )
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=morph_iterations)
    
    # Step 4: Sure background (dilate to expand background)
    sure_bg = cv2.dilate(opening, kernel, iterations=3)
    
    # Step 5: Sure foreground via distance transform
    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    
    # Handle case where dist_transform max is 0 (all black after opening)
    if dist_transform.max() == 0:
        # Fallback: return the opening result as mask
        return opening, np.zeros_like(gray, dtype=np.int32)
    
    ret, sure_fg = cv2.threshold(
        dist_transform, dist_threshold * dist_transform.max(), 255, 0
    )
    sure_fg = np.uint8(sure_fg)
    
    # Step 6: Unknown region
    unknown = cv2.subtract(sure_bg, sure_fg)
    
    # Step 7: Create markers
    ret, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1  # background becomes 1 instead of 0
    markers[unknown == 255] = 0  # unknown regions marked as 0
    
    # Step 8: Apply watershed
    # Watershed requires 3-channel BGR image
    img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    markers = cv2.watershed(img_bgr, markers)
    
    # Create binary mask
    # markers == 1 is background, markers == -1 is boundary
    # markers > 1 are foreground objects
    mask = np.where(markers > 1, 255, 0).astype(np.uint8)
    
    return mask, markers


# Test on single image
test_idx = 0
test_row = df_manifest.iloc[test_idx]

img_path = PREPROCESSED_DIR / 'images' / test_row['filename']
mask_path = PREPROCESSED_DIR / 'masks' / test_row['mask_filename']

test_img, test_gt = load_image_pair(img_path, mask_path)

# Test with default parameters
mask_ws, markers = segment_watershed(test_img, dist_threshold=0.5)

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(test_img)
axes[0].set_title('Input image')
axes[0].axis('off')

axes[1].imshow(mask_ws, cmap='gray')
axes[1].set_title('Watershed result')
axes[1].axis('off')

axes[2].imshow(markers, cmap='jet')
axes[2].set_title('Markers (colored)')
axes[2].axis('off')

axes[3].imshow(test_gt, cmap='gray')
axes[3].set_title('Ground truth')
axes[3].axis('off')

plt.suptitle(f'Watershed Test - {test_row["subset"]}', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'watershed_test.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"test complete on: {test_row['filename']}")

In [ ]:
# Part 3: Parameter Tuning - distance transform threshold

thresholds = [0.3, 0.5, 0.7]

fig, axes = plt.subplots(1, len(thresholds) + 1, figsize=(20, 4))

axes[0].imshow(test_img)
axes[0].set_title('input')
axes[0].axis('off')

for i, thr in enumerate(thresholds):
    mask, _ = segment_watershed(test_img, dist_threshold=thr)
    
    axes[i+1].imshow(mask, cmap='gray')
    axes[i+1].set_title(f'threshold = {thr}')
    axes[i+1].axis('off')

plt.suptitle('Watershed: Distance Threshold Comparison', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'threshold_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Part 3b: Quick quantitative check to pick best threshold

def evaluate_quick(pred_mask, gt_mask):
    """Quick IoU calculation."""
    pred = (pred_mask > 127).astype(np.uint8)
    gt = (gt_mask > 127).astype(np.uint8)
    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return intersection / union if union > 0 else 0.0

# Test on first 10 images
n_test = 10
thresholds = [0.2, 0.3, 0.5, 0.7]
kernel_sizes = [3, 5]

tuning_results = []

for idx in tqdm(range(min(n_test, len(df_manifest))), desc="tuning"):
    row = df_manifest.iloc[idx]
    img_path = PREPROCESSED_DIR / 'images' / row['filename']
    mask_path = PREPROCESSED_DIR / 'masks' / row['mask_filename']
    img, gt = load_image_pair(img_path, mask_path)
    
    for thr in thresholds:
        for ks in kernel_sizes:
            mask, _ = segment_watershed(img, dist_threshold=thr, morph_kernel_size=ks)
            iou = evaluate_quick(mask, gt)
            tuning_results.append({
                'threshold': thr, 'kernel_size': ks, 'iou': iou
            })

df_tuning = pd.DataFrame(tuning_results)
summary = df_tuning.groupby(['kernel_size', 'threshold'])['iou'].mean().unstack()
print("\nMean IoU across first 10 images:")
print(summary.round(4))
print(f"\nBest config: {df_tuning.groupby(['kernel_size', 'threshold'])['iou'].mean().idxmax()}")

In [ ]:
# Part 4: Batch Processing
# 
# >>> UPDATE WATERSHED_CONFIG BELOW BASED ON TUNING RESULTS <<<

WATERSHED_CONFIG = {
    'dist_threshold': 0.5,       # fraction of max distance - update based on tuning
    'morph_kernel_size': 3,      # kernel size - update based on tuning
    'morph_iterations': 2,
}

print("watershed configuration:")
for key, value in WATERSHED_CONFIG.items():
    print(f"  {key}: {value}")

# Process all images
print(f"\nprocessing {len(df_manifest)} images with watershed...")

results = []

for idx, row in tqdm(df_manifest.iterrows(), total=len(df_manifest), desc="watershed"):
    # Load image
    img_path = PREPROCESSED_DIR / 'images' / row['filename']
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Segment
    try:
        start_time = time.time()
        mask, _ = segment_watershed(
            img,
            dist_threshold=WATERSHED_CONFIG['dist_threshold'],
            morph_kernel_size=WATERSHED_CONFIG['morph_kernel_size'],
            morph_iterations=WATERSHED_CONFIG['morph_iterations']
        )
        elapsed = time.time() - start_time
        
        # Save result
        output_path = RESULTS_DIR / row['mask_filename']
        cv2.imwrite(str(output_path), mask)
        
        results.append({
            'filename': row['filename'],
            'mask_filename': row['mask_filename'],
            'subset': row['subset'],
            'status': 'success',
            'time_seconds': round(elapsed, 3),
            'output_path': str(output_path)
        })
        
    except Exception as e:
        print(f"\nerror processing {row['filename']}: {e}")
        results.append({
            'filename': row['filename'],
            'mask_filename': row['mask_filename'],
            'subset': row['subset'],
            'status': 'failed',
            'error': str(e)
        })

# Save results manifest
df_results = pd.DataFrame(results)
df_results.to_csv(RESULTS_DIR / 'watershed_results.csv', index=False)

print(f"\nprocessing complete:")
print(f"  success: {(df_results['status'] == 'success').sum()}")
print(f"  failed: {(df_results['status'] == 'failed').sum()}")
if 'time_seconds' in df_results.columns:
    print(f"  avg time per image: {df_results['time_seconds'].mean():.3f}s")
print(f"  results saved to: {RESULTS_DIR}")

In [ ]:
# Part 5: Qualitative Validation

n_samples_per_subset = 2
validation_samples = []

for subset in sorted(df_manifest['subset'].unique()):
    subset_data = df_results[df_results['subset'] == subset]
    successful = subset_data[subset_data['status'] == 'success']
    
    if len(successful) > 0:
        samples = successful.sample(min(n_samples_per_subset, len(successful)), random_state=42)
        validation_samples.extend(samples.index.tolist())

print(f"visualizing {len(validation_samples)} validation samples...")

for sample_idx in validation_samples:
    row = df_results.iloc[sample_idx]
    
    img_path = PREPROCESSED_DIR / 'images' / row['filename']
    gt_path = PREPROCESSED_DIR / 'masks' / row['mask_filename']
    pred_path = RESULTS_DIR / row['mask_filename']
    
    img, gt_mask = load_image_pair(img_path, gt_path)
    pred_mask = cv2.imread(str(pred_path), cv2.IMREAD_GRAYSCALE)
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    axes[0].imshow(img)
    axes[0].set_title('input')
    axes[0].axis('off')
    
    axes[1].imshow(pred_mask, cmap='gray')
    axes[1].set_title('watershed prediction')
    axes[1].axis('off')
    
    axes[2].imshow(gt_mask, cmap='gray')
    axes[2].set_title('ground truth')
    axes[2].axis('off')
    
    overlay = img.copy()
    overlay[pred_mask > 127] = overlay[pred_mask > 127] * 0.5 + np.array([0, 255, 0]) * 0.5
    axes[3].imshow(overlay.astype(np.uint8))
    axes[3].set_title('overlay (green=pred)')
    axes[3].axis('off')
    
    plt.suptitle(f'Validation - {row["subset"]} - {row["filename"][:50]}', fontsize=11)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'validation_{sample_idx}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Part 6: Save Metadata

metadata = {
    'method': 'watershed',
    'config': WATERSHED_CONFIG,
    'total_images': int(len(df_manifest)),
    'successful': int((df_results['status'] == 'success').sum()),
    'failed': int((df_results['status'] == 'failed').sum()),
    'timestamp': pd.Timestamp.now().isoformat()
}

with open(RESULTS_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("metadata saved")

In [ ]:
# Summary

print("\n" + "="*60)
print("WATERSHED SEGMENTATION COMPLETE")
print("="*60)
print(f"method: watershed")
print(f"dist threshold: {WATERSHED_CONFIG['dist_threshold']}")
print(f"kernel size: {WATERSHED_CONFIG['morph_kernel_size']}")
print(f"images processed: {len(df_manifest)}")
print(f"success rate: {(df_results['status'] == 'success').sum() / len(df_manifest) * 100:.1f}%")
print(f"results: {RESULTS_DIR}")
print("="*60)
print("\nNext step: run notebook 03 with METHOD_CONFIG name='watershed'")

## **Output Structure:**
```
results/watershed/
├── DIS-TE1_*.png              # 20 segmentation masks
├── DIS-TE2_*.png              # 20 segmentation masks
├── DIS-TE3_*.png              # 20 segmentation masks
├── DIS-TE4_*.png              # 20 segmentation masks
├── watershed_results.csv      # processing manifest
├── metadata.json              # configuration
├── watershed_test.png
├── threshold_comparison.png
└── validation_*.png
```